In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリの読み込み

import os
import shutil

import numpy as np
import optuna
from optuna.trial import Trial
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
# ディレクトリの定義

splited_dir = os.path.join("data", "splited")

In [ ]:
# モデルの読み込み

n_splits = 5

models = []

for i in range(1, n_splits + 1):
    model_path = os.path.join("runs", "exp", f"split_{i}", "weights", "best.pt")

    if os.path.exists(model_path):
        print(model_path)
        model = YOLO(model_path)
        models.append(model)

In [ ]:
# グリッドサーチ

param_grid = {
    "conf": np.unique(np.round(np.r_[0.001, np.arange(0.01, 0.301, 0.02)], 3)),
    "iou": np.round(np.arange(0.30, 0.901, 0.03), 3),
}

best_score = -1.0
best_params = None

for params in tqdm(ParameterGrid(param_grid)):
    conf = params["conf"]
    iou = params["iou"]

    scores = []

    for i in range(n_splits):
        results = models[i].val(
            data=os.path.join(splited_dir, f"split_{i + 1}", "data.yaml"),
            imgsz=640,
            batch=128,
            conf=conf,
            iou=iou,
            max_det=5,
            half=True,
            device=0,
            save=False,
        )

        scores.append(float(results.box.map))

    score = np.mean(scores)

    if score > best_score:
        best_score = score
        best_params = {"conf": conf, "iou": iou}

In [ ]:
# ベイズ最適化

def bound(
    center: float, lower: float, upper: float, radius: float
) -> tuple[float, float]:
    return max(lower, center - radius), min(upper, center + radius)


def objective(trial: Trial) -> float:
    lower, upper = bound(best_params["conf"], 0.001, 0.30, 0.02)
    conf = trial.suggest_float("conf", lower, upper)

    lower, upper = bound(best_params["iou"], 0.30, 0.90, 0.05)
    iou = trial.suggest_float("iou", lower, upper)

    scores = []

    for i in range(n_splits):
        results = models[i].val(
            data=os.path.join(splited_dir, f"split_{i + 1}", "data.yaml"),
            imgsz=640,
            batch=128,
            conf=conf,
            iou=iou,
            max_det=5,
            half=True,
            device=0,
            save=False,
        )

        scores.append(float(results.box.map))

    score = np.mean(scores)

    return score


sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.NopPruner()
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

study.optimize(objective, n_trials=80, show_progress_bar=True)

In [ ]:
# 結果の表示

print(study.best_value)
print(study.best_params)

In [ ]:
# フォルダの削除

shutil.rmtree(os.path.join("runs", "detect"))